# Workspace Preparation

This section guides how to setup the development environment.

### 1. Download and Install the required Softwares

- [Ollama](https://ollama.com/download) Open source LLM runner. (For local testing only) -- we use [vLLM](https://docs.vllm.ai/en/latest/) for production deployments.

- Docker [Windows](https://docs.docker.com/desktop/setup/install/windows-install/) or [Ubuntu](https://docs.docker.com/engine/install/ubuntu/) for the Development Container.

### 2. Ollama Server Preparation

- Pull the Ollama models we needed for this notebook.

```sh
# the LLM
ollama pull gemma4:e2b

# the embedding model for the RAG
ollama pull nomic-embed-text-v2-moe
```

- Run the Ollama server manually in CMD. (stop the ollama in windows icon tray first before running the below commands)

```sh
# this command will start the server in http://localhost:11434 of the host machine. 
# To access the server inside the docker container on windows, we will use host.docker.internal instead of localhost. http://host.docker.internal:11434
ollama serve
```

### 3. Development Container Preparation

#### 3.1 Create `Dockerfile` and save the following content.

```dockerfile
FROM python:3.11-slim

RUN apt-get update && apt-get install -y curl git && rm -rf /var/lib/apt/lists/* && \
  useradd -m agentuser

# 2. Install Playwright Python package
RUN pip install --no-cache-dir playwright

# 3. Install the browser AND all missing system dependencies (The Fix)
# This installs libgtk-3, libgdk-3, and all the .so files in your error log.
RUN playwright install chromium --with-deps

# 4. Setup your agent user
RUN useradd -m agentuser && \
    mkdir -p /home/agentuser/bin /home/agentuser/agent && \
    chown -R agentuser:agentuser /home/agentuser

USER agentuser
WORKDIR /home/agentuser/agent
```

#### 3.2 Create a `docker-compose.yml` and save the following content.

```yml
services:
  dev:
    build:
      dockerfile: Dockerfile
      context: .
    volumes:
      - ./:/home/agentuser/agent
    command: tail -f /dev/null
    network_mode: host
```

#### 3.3 Run the Development container.

```sh
docker compose up -d
```

### 4. Using the Development Container

#### 4.1 Open VSCode and attach to running container.

- press `ctrl+shift+P` and type in search `dev att`

#### 4.2 Create python virtual environment `venv` inside the container and install the required packages.

```sh
# run this command only once to create new virtual environment
python -m venv venv

# activate the environment
. ./venv/bin/activate

# install the packages
pip install langchain langgraph langchain_openai langchain_ollama

# freeze the package requirements so that we can re-install / upgrade later
pip freeze > requirements.txt

# to re-install
pip install -r requirements.txt

# to upgrade all
pip install -U -r requirements.txt
```

# Langchain

Is an open-source framework and orchestration environment designed to help developers build applications powered by Large Language Models (LLMs)

In this section we initialize the API client for LLM connection. We will also define the connection arguments to control the LLM text generation.

In [68]:
from langchain_core.messages import AIMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

# create client connection
# client = ChatOpenAI(
#   model='google/gemma-4-E4B-it',
#   base_url='http://vllmhost:8000/v1',
#   api_key='sample', # this is optional if locally served, can be any value or as declared in the LLM server, e.g. vLLM, Ollama
#   temperature=0,    # how creative is the LLM, the hyperparameter for random sampling.
#   max_tokens=4096,  # how long the output message can be
#   top_p=0.8,        # total probability mass, e.g. ['cat': 0.5, 'dog': 0.3, 'fish': 0.1, 'worm': 0.1] the LLM can only see 'cat' and 'dog'.
#   reasoning={
#     'effort': 'low',  # how deep the reasoning model think. low, medium, high
#   },
#   extra_body={
#       "tool_choice": "auto" # Explicitly tell vLLM to expect tools
#   }
# )

client = ChatOllama(
  model='gemma4:e2b',
  base_url='http://host.docker.internal:11434',
  api_key="ollama",
  temperature=0,
  max_tokens=4096,
  reasoning=False,
  thinking_level='low',
)

# send message
client.invoke("Hi!")


AIMessage(content='Hi! How can I help you today? 😊', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-06-05T16:17:37.2740179Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5710023700, 'load_duration': 5194088200, 'prompt_eval_count': 11, 'prompt_eval_duration': 40032700, 'eval_count': 11, 'eval_duration': 281112600, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019e9893-4f9d-76b2-87e4-3bb5dab91dd3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 11, 'total_tokens': 22})

# Utility: Get Text from AIMessage

When dealing with LLM running in different runner (e.g. Ollama, vLLM), We need utilities to parse the message content. There are different types of message in Langchain framework. All of them extends the `BaseMessage`.
- **SystemMessage**: defines the agent behavior, has the highest influence in LLM generation governing persona, constraints, and instructions.
- **HumanMessage**: the user input.
- **AIMessage**: the AI model response.
- **ToolMessage**: the `ToolNode` response, this message has unique `tool_call_id` that matches the items in AIMessage `tool_calls`.

In this section we will create the utility to parse the `AIMessage` text content.

In [69]:
# utility to get message content
def getTextContent(message: AIMessage):
  if isinstance(message.content, list):
    return "\n".join([
      m['text'] for m in message.content if 'text' in m
    ])
  return message.content

message = client.invoke("Hi!")
print(f'\nPARSED TEXT:\n{getTextContent(message)}')


PARSED TEXT:
Hi! How can I help you today? 😊


# Prompt Templates

In this section we define following prompt templates
- **ChatPromptTemplate.from_messages**: a template that can control the LLM persona and behavior `SystemMessage`. 
- **ChatPromptTemplate.from_template**: a template that automatically injects prompts to user input `HumanMessage`.

In [70]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt with system behavior
_comedian_prompt = ChatPromptTemplate.from_messages([
  SystemMessage(content="You are a proffessional Comedian."),
  MessagesPlaceholder(variable_name="messages")
])

comedian = _comedian_prompt | client
comedian_result = comedian.invoke({
  'messages': [
    HumanMessage(content="Tell me a short joke about cats")
  ]
})
print(getTextContent(comedian_result))
print('---')

# Helper prompt for HumanMessage
_comedian_prompt2 = ChatPromptTemplate.from_template("{input}\n\nIn tagalog. Do not include your introduction.")
commedian2 = _comedian_prompt2 | client
comedian_result2 = commedian2.invoke({
  'input': "Tell me a short joke about cats"
})
print(getTextContent(comedian_result2))


Why did the cat cross the road?

To see if it tasted like tuna on the other side! 🐟😼
---
Bakit mahilig magtago ang mga pusa?

Kasi gusto nilang maging misteryoso!


# Langchain Expression Language (LCEL)

In this section, we use the `|` operator to chain the invocation like a pipeline.

```py
model1 = system_prompt1 | llm_client
model2 = system_prompt2 | llm_client
chain = model1 | call_model2 | ... | output_parser
```
to chain the model1 output another invoke call, we need to create a function wrapper `call_model2` to pass-down the `AIMessage` from `model1` to `model2`.

In [71]:
# create new persona
_joke_critic_prompt = ChatPromptTemplate.from_messages([
  SystemMessage(content="You are a professional Quality Assurance for jokes created my comedians. Your task is to identify if the joke is funny or not. \n\nOUTPUT FORMAT:\nALWAYS mention the joke in your response."),
  MessagesPlaceholder(variable_name="messages")
])

_joke_critic = _joke_critic_prompt | client

# function wrapper for LCEL
def call_critic(message: AIMessage):
  return _joke_critic.invoke({
    'messages': [
      message
    ]
  })

chain = comedian | call_critic | getTextContent

print(chain.invoke({
  'messages': [
    HumanMessage(content="Tell me a short joke about cats")
  ]
}))


Joke: Why did the cat cross the road? To see if it tasted like tuna on the other side! 🐟😼

Quality Assurance Assessment: Not Funny. (The punchline is a weak pun/wordplay that doesn't land as a traditional joke.)


# Structured Output

LLM are smart enough to follow the instructions that describes how the expected output looks like. 

When using structured output, we define a class that extends the Pydantic BaseModel validator, this will automatically throw an error when the value assigned to property is a type mismatch. 

PS: The field description must be informative enough for the model to not hallucinate when assigning the value.

In [72]:
from typing import Literal, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
import warnings
import json

warnings.filterwarnings("ignore", ".*Pydantic serializer warnings.*", UserWarning)

# define the output structure class with pydantic validations
class SampleStructure(BaseModel):
  request_type: Literal['casual','needs_planning'] = Field(description="The user request type. casual | needs_planning")
  message: str = Field(description="message to the user.")

# bind the structure
model = client.with_structured_output(SampleStructure)

# system instruction for the llm output structure
system_prompt = """
RESPONSE FORMAT
{
  "request_type": string, // the user request type. casual | needs_planning
  "message": string // message to the user.
}
""".strip()

response = model.invoke([
  SystemMessage(content=system_prompt),
  HumanMessage(content="Can you check my timesheet this week?")
])

if isinstance(response, SampleStructure):
  print(response)
  print(response.model_dump_json())


request_type='needs_planning' message='I would be happy to check your timesheet, but I need you to provide it to me first. Please share the timesheet you are referring to.'
{"request_type":"needs_planning","message":"I would be happy to check your timesheet, but I need you to provide it to me first. Please share the timesheet you are referring to."}


# Utility: Genenerate LLM Schema

In this section we create a utility function that can generate the system prompt from the Pydantic BaseModel schema.

We also add the optional parameter `additional_constraints` so we can influence the model output.

In [73]:
from typing import Any, Dict


def generate_llm_schema(model: type[BaseModel], escape_chars: bool = True, additional_constraints: list[str] = []) -> str:
    """Converts a Pydantic model into a clean text format optimized for LLMs, 
    supporting deeply nested models and definitions.
    """

    schema = model.model_json_schema()
    # Pydantic v2 stores shared nested models in '$defs'
    defs = schema.get("$defs", {})

    def resolve_ref(prop: Dict[str, Any]) -> Dict[str, Any]:
        """Helper to resolve JSON schema $ref pointers to actual definitions."""
        if "$ref" in prop:
            ref_name = prop["$ref"].split("/")[-1]
            return defs.get(ref_name, prop)
        return prop

    def parse_property_type(prop: Dict[str, Any]) -> str:
        """Recursively parses a property dictionary into a string representation."""
        prop = resolve_ref(prop)
        
        if "anyOf" in prop:
            types = []
            for sub_prop in prop["anyOf"]:
                sub_prop = resolve_ref(sub_prop)
                if "type" in sub_prop:
                    types.append(parse_property_type(sub_prop))
                elif "$ref" in sub_prop:
                    types.append(parse_property_type(sub_prop))
            return " | ".join(filter(None, types)) or "any"
        
        field_type = prop.get("type", "any")

        # Handle Deeply Nested Object
        if field_type == "object" and "properties" in prop:
            nested_lines = ["{"]
            for f_name, f_props in prop["properties"].items():
                f_props = resolve_ref(f_props)
                f_type = parse_property_type(f_props)
                f_desc = f_props.get("description", "")
                desc_suffix = f"  // {f_desc}" if f_desc else ""
                nested_lines.append(f'  "{f_name}": {f_type},{desc_suffix}')
            nested_lines.append("}")
            return "\n".join(nested_lines)

        # Handle Array of primitives or nested objects
        elif field_type == "array":
            items_prop = resolve_ref(prop.get("items", {}))
            items_type = parse_property_type(items_prop)
            
            # Formatting block arrays nicely if they are nested objects
            if "\n" in items_type:
                # Indent nested object arrays gracefully
                indented_items = items_type.replace("\n", "\n  ")
                return f"Array<\n  {indented_items}\n>"
            return f"Array<{items_type}>"

        return field_type

    # Start parsing the top-level schema
    lines = ["{"]
    for field_name, properties in schema.get("properties", {}).items():
        properties = resolve_ref(properties)
        field_type = parse_property_type(properties)
        description = properties.get("description", "")
        
        desc_suffix = f"  // {description}" if description else ""
        
        # If the returned type is a nested block, indent its inner contents nicely
        if "\n" in field_type:
            indented_type = field_type.replace("\n", "\n  ")
            lines.append(f'  "{field_name}": {indented_type},{desc_suffix}')
        else:
            lines.append(f'  "{field_name}": {field_type},{desc_suffix}')
            
    lines.append("}")
    
    final_output = "\n".join(lines)
    
    if escape_chars:
        # Escapes { to {{ and } to }} so it can safely be fed into .format() or f-strings
        final_output = final_output.replace('{', '{{').replace('}', '}}')

    # compute the constraints
    constraints = '\n'.join([ 
        f'- {item}' 
        for item in ["Do not use markdown formatting in your response. Respond only with the raw JSON format as specified."] + additional_constraints
    ])

    # the final output format for system prompt
    return (
       "# CONSTRAINTS\n"
       f"{constraints}\n\n"
       "# RESPONSE FORMAT\n"
       f"{final_output}"
    )

class _Sample(BaseModel):
    prop1: str = Field("description of property 1")
    prop2: int = Field("description of property 2")

print(generate_llm_schema(_Sample, additional_constraints=[
    # "If condition for constraint 1",
    # "If condition for constraint 2"
]))

# CONSTRAINTS
- Do not use markdown formatting in your response. Respond only with the raw JSON format as specified.

# RESPONSE FORMAT
{{
  "prop1": string,
  "prop2": integer,
}}


# Influencing the LLM output by using Constraints

In this section we will try to control the LLM decision making for each property value in SampleStructure output format.

In [74]:
_system_output_format = generate_llm_schema(SampleStructure, additional_constraints=[
  # add constraints here to influence the output
  'if request_type is casual, you need to be friendly respond with emoji.',
  "if request_type is needs_planning, message=''"
])

print(_system_output_format)

response = model.invoke([
  SystemMessage(content=_system_output_format),
  HumanMessage(content="can you check my timesheet today")
])

if isinstance(response, SampleStructure):
  print(response)


# CONSTRAINTS
- Do not use markdown formatting in your response. Respond only with the raw JSON format as specified.
- if request_type is casual, you need to be friendly respond with emoji.
- if request_type is needs_planning, message=''

# RESPONSE FORMAT
{{
  "request_type": string,  // The user request type. casual | needs_planning
  "message": string,  // message to the user.
}}
request_type='needs_planning' message=''


# Prompt Engineering

LLM generates the output based on user input. The context from the user input is like a magnet for each tokens in LLM training experience, meaning if the context that starts the conversation was not strong enough to attract the tokens to build the correct response, the quality of the LLM output will suffer from hallucination.

In this section we will create a model that validates the user input against the whitelisted topics. We will use a `SystemMessage` based on inverted pyramid / funnel pattern.
```txt
\ -------  persona, role, task ------- /   - Core Instructions
 \ ---------- history data ---------- /    - Primary Context and Data 
  \ -- formatting and constraints -- /     - Formatting, Tone and Edge Cases
```

In [75]:
class _RedlineResult(BaseModel):
  request_type: Literal['casual', 'needs_planning', 'violation'] = Field(description="The user request type. casual | needs_planning | violation")
  message: str = Field(description="The message to the user.")
  reformulated_request: str = Field(description="The user request translated to english language.")

_response_format = generate_llm_schema(_RedlineResult, additional_constraints=[
  "if the request_type is 'casual', message='a friendly message with emoji', reformulated_request=''",
  "if the request_type is 'violation', message='explain why, do not use exclamations', reformulated_request=''",
  "if the request_type is 'needs_planning', message='', reformulated_request='translate the user request to english language to hightlight the intent'.",
])

_REDLINE_INSTRUCTIONS="""
# ROLE
You are a Policy enforcer. Your primary task is to validate the user input against the WHITELISTED_TOPICS and REDLINES. 
 
# WHITELISTED_TOPICS
- timesheet, attendance
- policies, handbook

# REDLINES
- NOT in WHITELISTED_TOPICS
- salary or payroll related

# REQUEST_TYPE
- **casual**: not needs_planning and not violation. (e.g. greetings)
- **needs_planning**: needs procedural steps. (e.g. data retrieval)
- **violation**: violates the REDLINES.

{response_format}
""".strip()

_prompt = ChatPromptTemplate.from_messages([
  # SystemMessage(content=_REDLINE_INSTRUCTIONS.format(response_format=_response_format)),
  ('system', _REDLINE_INSTRUCTIONS),
  MessagesPlaceholder(variable_name='messages')
])

print(_REDLINE_INSTRUCTIONS)
print('---')

_model = _prompt | client.with_structured_output(_RedlineResult)

[
  _model.invoke({
    'response_format': _response_format,
    'messages': [HumanMessage(content=user_input)]
  })
  for user_input in [
    "can you check my timesheet today",
    "hi, ano pangalan mo",
    "magkano sweldo ni Tolits?"
  ]
]

# ROLE
You are a Policy enforcer. Your primary task is to validate the user input against the WHITELISTED_TOPICS and REDLINES. 

# WHITELISTED_TOPICS
- timesheet, attendance
- policies, handbook

# REDLINES
- NOT in WHITELISTED_TOPICS
- salary or payroll related

# REQUEST_TYPE
- **casual**: not needs_planning and not violation. (e.g. greetings)
- **needs_planning**: needs procedural steps. (e.g. data retrieval)
- **violation**: violates the REDLINES.

{response_format}
---


[_RedlineResult(request_type='needs_planning', message='', reformulated_request='translate the user request to english language to hightlight the intent'),
 _RedlineResult(request_type='casual', message='Hello! I am a policy enforcer. 👋', reformulated_request=''),
 _RedlineResult(request_type='violation', message='requests regarding salary are not allowed', reformulated_request='')]

# ToolCall: Anatomy

In this section, we explore the core concept how LLM make tool calls.

The LLM make tool_calls by using the **BaseChatModel._bind_tools**, when we call the `.invoke()` the specific runner subclass (ChatOpenAI, ChatOllama, etc..) intercept the tools parameter duing payload generation for the model response.

As a result the `AIMessage` result will contain the list of `tool_calls` generated by the LLM.

In [76]:
from langchain_core.messages import ToolMessage

# the function to call
def _add_tool(a: int, b: int):
  return a + b

# the schema to bind in model using the langchain API
_ADD_TOOL_SCHEMA = {
  "type": "function",
  "function": {
    "name": "add",
    "description": "Calculates the sum of two value",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {"type": "number", "description": "the first value"},
        "b": {"type": "number", "description": "the first value"},
      },
      "required": ["a", "b"]
    }
  }
}

# tool schema binding
_model = client.bind_tools([_ADD_TOOL_SCHEMA])

# LLM request
_response = _model.invoke("what is the sum of 1 and 2")
print(_response)

# ToolNode execution, we display this part as dictionary to show how it works internally in ToolNode
_tools = {
  'add': _add_tool
}

# the ToolNode executes each request in tool_calls and return a ToolMessage, the tool_call_id is used to match the execution result to each tool.
[
  ToolMessage(name=request['name'], tool_call_id=request['id'], content=f"{_tools[request['name']](**request['args'])}")
  for request in _response.tool_calls
  if request['name'] in _tools and request['type'] == 'tool_call'
]


content='' additional_kwargs={} response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-06-05T16:17:50.4408284Z', 'done': True, 'done_reason': 'stop', 'total_duration': 696390200, 'load_duration': 269620400, 'prompt_eval_count': 94, 'prompt_eval_duration': 50066600, 'eval_count': 15, 'eval_duration': 370441700, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'} id='lc_run--019e9893-96ae-7a30-8698-2df7a9ceb6d2-0' tool_calls=[{'name': 'add', 'args': {'a': 1, 'b': 2}, 'id': '74537ff2-e604-456b-a834-b78677702e57', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 94, 'output_tokens': 15, 'total_tokens': 109}


[ToolMessage(content='3', name='add', tool_call_id='74537ff2-e604-456b-a834-b78677702e57')]

# ToolCall: Langchain @tool Decorator

In this section, instead of hardcoding the tool schema manually, we will use the Langchain builtin `@tool` decorator.

We also explore how the LLM process multiple tool call execution by reading the message history.

In [77]:
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage

# the function to call
@tool
def _add_tool(a: int, b: int):
  """Use this tool to add two numbers"""
  return a + b

@tool
def _minus_tool(a: int, b:int):
  """Use this tool to subtract two numbers"""
  return a - b

_tools = {
  '_add_tool': _add_tool,
  '_minus_tool': _minus_tool
}

_model = client.bind_tools(_tools.values())

# the state of our message history
_history:list[BaseMessage] = []

def _print_history():
  print([ m.__class__.__name__ for m in _history])

# initial state, the user will provide the input
# in this state we can also include the SystemMessage if the model has certain persona or procedure to follow.
print("** Initial **")
_print_history()

print("** Iteration 1 **")
_history.append(HumanMessage("what is the total of add 1 + 2 - 4"))
_response = _model.invoke(_history)
_history.append(_response)
_print_history()

print("** Tool Call 1 **")
_results = [
  ToolMessage(name=request['name'], tool_call_id=request['id'], content=f"{_tools[request['name']].invoke(request)}")
  for request in _response.tool_calls
  if request['name'] in _tools and request['type'] == 'tool_call'
]
_history.extend(_results)
_print_history()

print("** Iteration 2 **")
_response = _model.invoke(_history)
_history.append(_response)
_print_history()

print("** Tool Call 2 **")
_results = [
  ToolMessage(name=request['name'], tool_call_id=request['id'], content=f"{_tools[request['name']].invoke(request)}")
  for request in _response.tool_calls
  if request['name'] in _tools and request['type'] == 'tool_call'
]
_history.extend(_results)
_print_history()

print ("\n----RAW HISTORY---\n")
print("\n\n".join([ f"{m.__class__.__name__} ( {m} )" for m in _history]))

** Initial **
[]
** Iteration 1 **
['HumanMessage', 'AIMessage']
** Tool Call 1 **
['HumanMessage', 'AIMessage', 'ToolMessage']
** Iteration 2 **
['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']
** Tool Call 2 **
['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'ToolMessage']

----RAW HISTORY---

HumanMessage ( content='what is the total of add 1 + 2 - 4' additional_kwargs={} response_metadata={} )

AIMessage ( content='' additional_kwargs={} response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-06-05T16:17:51.178308Z', 'done': True, 'done_reason': 'stop', 'total_duration': 694181600, 'load_duration': 211134300, 'prompt_eval_count': 140, 'prompt_eval_duration': 57872500, 'eval_count': 17, 'eval_duration': 417088400, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'} id='lc_run--019e9893-9992-7361-b7c2-2aa83a011f4e-0' tool_calls=[{'name': '_add_tool', 'args': {'a': 1, 'b': 2}, 'id': 'd9533282-c370-4caf-a2c3-7290272caa0f', 'type': 'tool_ca